# 05 — Exploratory Data Analysis (Step 1.3)

Explores the cleaned Food-11 manifest (16,020 unique images, final stratified splits) and checks how the curated dish dataset relates to it. This notebook only reads existing files; cleaning is documented in notebooks 02–04. Figures are saved in `reports/figures/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

PROJECT_ROOT = Path.cwd().parent   # this notebook lives in notebooks/
fig_dir = PROJECT_ROOT / "reports" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "food11" / "food11_clean_manifest.csv")
print("Manifest shape:", manifest.shape)

## 1. Class distribution (final splits)

In [ ]:
# Numbers first: class x final split
counts = pd.crosstab(manifest["category"], manifest["new_split"])
counts = counts[["training", "validation", "evaluation"]]
counts["total"] = counts.sum(axis=1)
counts["train_%"] = (counts["training"] / counts["total"] * 100).round(1)
display(counts.sort_values("total", ascending=False))
print(counts[["training", "validation", "evaluation", "total"]].sum())

# Plot: final class distribution
order = counts.sort_values("total", ascending=False).index

plt.figure(figsize=(14, 6))
sns.countplot(
    data=manifest, x="category", hue="new_split",
    order=order, hue_order=["training", "validation", "evaluation"]
)
plt.title("Food-11 Final Class Distribution by Split (16,020 unique images)")
plt.xlabel("Food Category")
plt.ylabel("Number of Images")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Final split")
plt.tight_layout()
plt.savefig(fig_dir / "food11_class_distribution_final.png", dpi=150)
plt.show()

**Findings**

- 16,020 images: 9,612 training / 3,204 validation / 3,204 evaluation.
- Stratification worked: every class is 60% training (Rice 60.1% from rounding), and validation and evaluation differ by at most 1 image per class.
- Class imbalance: Dessert (2,391 images) vs Rice (471) is about 5:1. Evaluation should report macro-F1 as well as accuracy, and class weights are worth considering.

## 2. Image dimensions, formats and file sizes

In [ ]:
print("Formats:", manifest["format"].value_counts().to_dict())
print("Modes:  ", manifest["mode"].value_counts().to_dict())
print()
display(manifest[["width", "height", "aspect_ratio", "file_size_kb"]].describe().round(2))

# Most common image sizes
size_counts = (
    manifest.assign(size=manifest["width"].astype(str) + "x" + manifest["height"].astype(str))
    ["size"].value_counts()
)
print("\nNumber of distinct sizes:", len(size_counts))
display(size_counts.head(8))

# How extreme are the outliers?
print("Width > 512:        ", (manifest["width"] > 512).sum())
print("Height > 512:       ", (manifest["height"] > 512).sum())
print("Width or height > 1024:", ((manifest["width"] > 1024) | (manifest["height"] > 1024)).sum())
print("File size > 500 KB: ", (manifest["file_size_kb"] > 500).sum())
print("Smaller than 224 px (either side):", ((manifest["width"] < 224) | (manifest["height"] < 224)).sum())
print("99th percentile file size (KB):", manifest["file_size_kb"].quantile(0.99).round(1))

In [ ]:
# Log axes: a few huge images would squash linear plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(manifest["width"], manifest["height"], s=6, alpha=0.2)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_title("Image width vs height (log scale)")
axes[0].set_xlabel("Width (px)"); axes[0].set_ylabel("Height (px)")

sns.histplot(manifest["aspect_ratio"], bins=40, ax=axes[1])
axes[1].set_title("Aspect ratio distribution (width / height)")
axes[1].set_xlabel("Aspect ratio")

sns.histplot(manifest["file_size_kb"], bins=50, log_scale=True, ax=axes[2])
axes[2].set_title("File size distribution (log scale)")
axes[2].set_xlabel("File size (KB, log scale)")

plt.tight_layout()
plt.savefig(fig_dir / "food11_image_dimensions.png", dpi=150)
plt.show()

**Findings**

- All 16,020 images are JPEG in RGB mode.
- 512×512 is the most common size (9,283 images, 57.9%). The 8 most common sizes cover 84.7%, but there are 825 distinct sizes in total.
- Aspect ratio is dominated by square images, followed by 4:3 landscape and 3:4 portrait.
- Outliers are rare: 907 images are wider than 512 px (5.7%), 197 have a side above 1,024 px (1.2%), and 171 files exceed 500 KB (1.1%). The largest image is 9,542 px wide and the largest file is about 9.9 MB.
- Only 2 images have a side below 224 px.
- Consequence: resizing to a fixed model input size is enough, with no outlier removal needed. Log axes are used because the outliers squash linear plots.

## 3. Visual diversity

In [ ]:
classes = sorted(manifest["category"].unique())
N_ROWS = 3

fig, axes = plt.subplots(N_ROWS, len(classes), figsize=(len(classes) * 2, N_ROWS * 2.3))

for col, cls in enumerate(classes):
    sample = manifest[manifest["category"] == cls].sample(N_ROWS, random_state=42)
    for row_i, (_, rec) in enumerate(sample.iterrows()):
        ax = axes[row_i, col]
        img = Image.open(rec["file_path"]).convert("RGB")
        img.thumbnail((256, 256))
        ax.imshow(img)
        ax.set_aspect("equal")
        ax.set_xlim(img.width / 2 - 128, img.width / 2 + 128)
        ax.set_ylim(img.height / 2 + 128, img.height / 2 - 128)
        ax.axis("off")
        if row_i == 0:
            ax.set_title(cls, fontsize=9, fontweight="bold")

plt.suptitle("Food-11: random samples per class", fontsize=13)
plt.tight_layout()
plt.savefig(fig_dir / "food11_sample_grid.png", dpi=100)
plt.show()

**Findings**

- The images are real-world photos with varied lighting, plating, backgrounds and camera angles, so within-class variety is high (for example Bread and Soup).
- Some samples are weak or noisy: a market stall in Vegetable-Fruit, a butter tub in Dairy product, a dark low-contrast Dessert photo.
- Plausible confusions are Noodles-Pasta vs Rice, Fried food vs Bread, and Meat vs Fried food. These are hypotheses from 3 samples per class and need a confusion matrix to confirm.
- Food-11 provides dish-level labels only, with no ingredients and no text.

## 4. Image–text association check (current limitation)

In [ ]:
curated_dir = PROJECT_ROOT / "data" / "processed" / "curated"
for name in ["dishes", "ingredients", "ingredient_reference", "dish_nutrition"]:
    df = pd.read_csv(curated_dir / f"{name}.csv")
    print(f"=== {name}.csv  shape={df.shape}")
    print("Columns:", df.columns.tolist())
    display(df.head(4))

In [ ]:
dishes = pd.read_csv(curated_dir / "dishes.csv")

mapping = {"Rice-based": "Rice", "Egg-based": "Egg", "Pasta": "Noodles-Pasta"}
dishes["food11_class"] = dishes["category"].map(mapping)
dishes["link_status"] = dishes["food11_class"].apply(
    lambda c: "category-level (heuristic)" if pd.notna(c) else "no matching Food-11 class"
)
dishes.to_csv(curated_dir / "dish_food11_mapping.csv", index=False)
display(dishes)

available = manifest["category"].value_counts()
summary = (
    dishes.dropna(subset=["food11_class"])
    .groupby("food11_class")
    .agg(n_dishes=("dish_id", "count"))
)
summary["food11_images"] = summary.index.map(available)
display(summary)
print("Dishes with a Food-11 link:", dishes["food11_class"].notna().sum(), "of", len(dishes))

**Findings**

- Food-11 has images and class labels but no text. The curated dataset has dishes, ingredients and USDA references but no images or text.
- No image–text pairs exist yet, so correct association cannot be verified at this stage.
- A coarse category-level mapping is saved in `data/processed/curated/dish_food11_mapping.csv`: 7 of 10 dishes link to a Food-11 class (Rice: 3 dishes / 471 images, Egg: 2 / 1,643, Noodles-Pasta: 2 / 733). The 3 Potato-based and Mixed dishes have no matching class.
- This mapping is a heuristic: it says "same kind of dish", not "this image shows this dish".
- Paired image–text data will be constructed in a later stage.